In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    auc
)

In [4]:
# Step 2 — Load Training & Holdout Datasets

TRAIN_PATH = "cell2celltrain.csv"
HOLDOUT_PATH = "cell2cellholdout.csv"

train = pd.read_csv(TRAIN_PATH)
holdout = pd.read_csv(HOLDOUT_PATH)

print("Train shape:", train.shape)
print("Holdout shape:", holdout.shape)
print(train.head())


Train shape: (51047, 58)
Holdout shape: (20000, 58)
   CustomerID Churn  MonthlyRevenue  MonthlyMinutes  TotalRecurringCharge  \
0     3000002   Yes           24.00           219.0                  22.0   
1     3000010   Yes           16.99            10.0                  17.0   
2     3000014    No           38.00             8.0                  38.0   
3     3000022    No           82.28          1312.0                  75.0   
4     3000026   Yes           17.14             0.0                  17.0   

   DirectorAssistedCalls  OverageMinutes  RoamingCalls  PercChangeMinutes  \
0                   0.25             0.0           0.0             -157.0   
1                   0.00             0.0           0.0               -4.0   
2                   0.00             0.0           0.0               -2.0   
3                   1.24             0.0           0.0              157.0   
4                   0.00             0.0           0.0                0.0   

   PercChangeRevenues 

In [5]:
# Step 3 - Column Types
target_col = "Churn"

cat_cols = train.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in train.columns if c not in cat_cols + [target_col]]

print("Categorical columns:", cat_cols)
print("First few numeric columns:", num_cols[:10])

Categorical columns: ['Churn', 'ServiceArea', 'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'Homeownership', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 'NotNewCellphoneUser', 'OwnsMotorcycle', 'HandsetPrice', 'MadeCallToRetentionTeam', 'CreditRating', 'PrizmCode', 'Occupation', 'MaritalStatus']
First few numeric columns: ['CustomerID', 'MonthlyRevenue', 'MonthlyMinutes', 'TotalRecurringCharge', 'DirectorAssistedCalls', 'OverageMinutes', 'RoamingCalls', 'PercChangeMinutes', 'PercChangeRevenues', 'DroppedCalls']


In [6]:
# Step 4 - Filling missing values

num_medians = train[num_cols].median()
cat_modes = train[cat_cols].mode().iloc[0]

# Apply to TRAIN
train[num_cols] = train[num_cols].fillna(num_medians)
train[cat_cols] = train[cat_cols].fillna(cat_modes)

# Apply to HOLDOUT
holdout[num_cols] = holdout[num_cols].fillna(num_medians)
holdout[cat_cols] = holdout[cat_cols].fillna(cat_modes)

print("Missing values in train:", train.isnull().sum().sum())
print("Missing values in holdout:", holdout.isnull().sum().sum())

Missing values in train: 0
Missing values in holdout: 0


In [7]:
# Step 5 — Encode Categorical Columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    combined = pd.concat([train[col], holdout[col]], axis=0)
    le.fit(combined)

    train[col] = le.transform(train[col])
    holdout[col] = le.transform(holdout[col])

    encoders[col] = le

train.head()

,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,1,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,0,0,8,1,0,2,4,0
1,3000010,1,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,0,0,8,0,3,2,4,2
2,3000014,0,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,0,0,15,0,2,3,1,2
3,3000022,0,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,...,0,6,0,0,0,0,3,0,3,0
4,3000026,1,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,...,0,9,0,1,0,0,0,0,4,2


In [8]:
# Section 6 - Drop cols

drop_cols = ["CustomerID", "ServiceArea", "Occupation", "PrizmCode", "NewCellphoneUser"]

X_full = train.drop(columns=[target_col] + [c for c in drop_cols if c in train.columns])
y_full = train[target_col]

X_holdout = holdout.drop(columns=[target_col] + [c for c in drop_cols if c in holdout.columns])

print("Final training features:", X_full.shape)
print("Final holdout features:", X_holdout.shape)

Final training features: (51047, 52)
Final holdout features: (20000, 52)


In [9]:
# Step 7 - validation

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

rf_temp = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_temp.fit(X_tr, y_tr)

val_pred = rf_temp.predict(X_val)
val_proba = rf_temp.predict_proba(X_val)[:, 1]

print("\n=== INTERNAL VALIDATION ===")
print("Accuracy:", round(accuracy_score(y_val, val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, val_proba), 4))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred))
print("Classification Report:\n", classification_report(y_val, val_pred))



=== INTERNAL VALIDATION ===
Accuracy: 0.7201
ROC AUC: 0.6617
Confusion Matrix:
 [[7120  148]
 [2710  232]]
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.98      0.83      7268
           1       0.61      0.08      0.14      2942

    accuracy                           0.72     10210
   macro avg       0.67      0.53      0.49     10210
weighted avg       0.69      0.72      0.63     10210



In [10]:
# Step 8 — Train final Model Using All Training Data

rf_final = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X_full, y_full)
print("Final model trained on full dataset.")

Final model trained on full dataset.


In [11]:
# Step 9 — Predict on HOLDOUT Dataset

holdout_pred = rf_final.predict(X_holdout)
holdout_proba = rf_final.predict_proba(X_holdout)[:, 1]

holdout["Predicted_Churn"] = holdout_pred
holdout["Churn_Probability"] = holdout_proba

holdout[["CustomerID", "Predicted_Churn", "Churn_Probability"]].head()

,CustomerID,Predicted_Churn,Churn_Probability
0,3000006,0,0.306667
1,3000018,0,0.280000
2,3000034,0,0.353333
3,3000070,0,0.206667
4,3000074,0,0.226667


In [14]:
def risk_label(p):
    if p >= 0.60:
        return "High Risk"
    elif p >= 0.30:
        return "Medium Risk"
    else:
        return "Low Risk"
        
holdout["Risk_Level"] = holdout["Churn_Probability"].apply(risk_label)

# Display sample
holdout[["CustomerID", "Predicted_Churn", "Churn_Probability", "Risk_Level"]].head(20)

output = holdout[["CustomerID", "Predicted_Churn", "Churn_Probability", "Risk_Level"]]
output.to_csv("holdout_predictions.csv", index=False)

print("Saved: holdout_predictions.csv")

Saved: holdout_predictions.csv
